# Atari Pong with frame preprocessing

This example uses standard [Gymnasium Atari (ALE)](https://gymnasium.farama.org/environments/atari/) envs — mouse-env does not change the games and has no Atari-specific code. Atari is configured with two general `EnvConfig` knobs: build the env in an `env_fn` factory (`gym.make` + `gymnasium.wrappers.AtariPreprocessing`), and set `observation_kind="image"` to route the frame to the image channel. Register ALE yourself with `gymnasium.register_envs(ale_py)`.

## Why `observation_kind="image"` is required

mouse-env auto-detects the observation channel from the Gymnasium observation space. Most spaces are self-describing: a `Box(float32)` is continuous, a `Discrete(n)` is discrete. Image observations are the exception: after `AtariPreprocessing`, the space is a `Box(uint8, shape=(84, 84))` — which looks identical to a 2-D discrete observation. Auto-detection cannot distinguish between the two.

Setting `observation_kind="image"` explicitly tells mouse-env to route the observation to the image channel and preserve its 2-D shape. Without it, the observation would be misclassified and the shape would be wrong.

The three valid values for `observation_kind` are:

| Value | When to use |
|-------|-------------|
| `None` (default) | Let mouse-env auto-detect from the observation space (works for most envs) |
| `"continuous"` | Force continuous (float vector) channel |
| `"discrete"` | Force discrete (integer scalar) channel |
| `"image"` | Force image channel; required for uint8 Box observations like Atari frames |

## Install extra

Requires `pip install 'mouse-env[atari]'`, which installs `ale_py` (ALE ROMs) and `opencv-python` (needed by `AtariPreprocessing` for frame resizing).

In [ ]:
import ale_py
import gymnasium as gym
import numpy as np
from gymnasium.wrappers import AtariPreprocessing

from mouse_envs import EnvConfig, make_env

gym.register_envs(ale_py)

## Configuration

`frameskip=1` in the factory disables the base env's built-in frame skipping so `AtariPreprocessing` owns it entirely — a common pattern when using `AtariPreprocessing`.

In [ ]:
def make_pong():
    env = gym.make("ALE/Pong-v5", frameskip=1, max_episode_steps=10000)
    return AtariPreprocessing(env, frame_skip=4, screen_size=84, noop_max=30)


cfgs = [
    EnvConfig(
        id="ALE/Pong-v5",
        reset_seed=i,
        episodes_per_task=5,
        observation_kind="image",
        env_fn=make_pong,
    )
    for i in range(4)
]
env = make_env(cfgs)

## First step and observation shape

After preprocessing, each env emits an 84×84 frame in its native shape. `env.obs_key` tells you which observation channel to read.

In [ ]:
outputs = env.step(env.sample_random_inputs())

print(f"obs spec:  {env.output_specs[0].observation}")
batch_obs = np.stack([r["observation"].numpy() for r in outputs])
print(f"obs shape: {batch_obs.shape}")  # (env.num_envs, 84, 84)

## Show a preprocessed frame

Each observation is already an 84×84 grayscale frame — exactly what the agent sees after `AtariPreprocessing`.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, env.num_envs, figsize=(3 * env.num_envs, 3))
for ax, frame in zip(np.atleast_1d(axes), batch_obs):
    ax.imshow(frame, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")
fig.tight_layout()
plt.show()

## Short follow-up rollout

In [ ]:
for _ in range(10):
    env.step(env.sample_random_inputs())

env.close()